In [ ]:
import torch
torch.cuda.is_available()
!/home/student/sky-scan/venv/bin/python -m pip install --upgrade pip

In [ ]:
!pip install opencv-python torch
!pip install tensorflow numpy matplotlib


In [ ]:
import torch

# Check if CUDA is available
if torch.cuda.is_available():
    print(f"✅ Running on GPU: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️ No GPU detected, running on CPU.")

## 1. Generate tiles image

In [5]:
base_path = "/home/student/sky-scan/data"


from PIL import Image
import os

def get_all_char_before_dot(s):
    dot_index = s.find('.')
    if dot_index > 0:
        return s[:dot_index]
    else:
        return None  # Return None if there's no character before the dot

def generate_tiles(image_path, tile_width, tile_height, step, output_folder):
    # Open the image
    image = Image.open(image_path)
    image_width, image_height = image.size

    # Generate tiles
    for x in range(0, image_width - tile_width + 1, step):
        for y in range(0, image_height - tile_height + 1, step):
            # Define the bounding box for the current tile
            left = x
            upper = y
            right = left + tile_width
            lower = upper + tile_height
            bbox = (left, upper, right, lower)

            # Crop the image to the bounding box to create the tile
            tile = image.crop(bbox)

            # Save the tile to the output folder
            image_file_name=os.path.basename(image_path)
            image_idx = get_all_char_before_dot(image_file_name)
            tile.save(f"{output_folder}/{image_idx}_{x}_{y}.png") #image_id_x_y.png

    print(f"✅ Tiles from {image_file_name} are generated successfully.")

def generate_binary_tiles(image_path, tile_width, tile_height, step, output_folder):
    # Open the image in grayscale mode (black & white)
    image = Image.open(image_path).convert("L")  # "L" mode ensures grayscale (0-255)
    image_width, image_height = image.size

    # Generate tiles
    for x in range(0, image_width - tile_width + 1, step):
        for y in range(0, image_height - tile_height + 1, step):
            # Define the bounding box for the current tile
            left = x
            upper = y
            right = left + tile_width
            lower = upper + tile_height
            bbox = (left, upper, right, lower)

            # Crop the image to create the tile
            tile = image.crop(bbox)

            # Convert to binary (0 or 255) to ensure black and white format
            tile = tile.point(lambda p: 255 if p > 127 else 0, mode="1")  # Thresholding

            # Save the tile to the output folder
            image_file_name = os.path.basename(image_path)
            image_idx = image_file_name.split('.')[0]  # Get filename without extension
            tile.save(f"{output_folder}/{image_idx}_{x}_{y}.png", format="PNG")  # Save as binary PNG

    print(f"✅ Binary tiles from {image_file_name} are generated successfully.")


In [6]:
# Generate tiles for original images
os.makedirs(f'{base_path}/patch', exist_ok=True)

for i in range(1,13): #12 images
    generate_tiles(f'{base_path}/Tiles/{i}.jpg', 256, 256, 128, f'{base_path}/patch')

print("Original images are generated successfully.")

# Generate tiles for binary mask images
os.makedirs(f'{base_path}/patch-binary', exist_ok=True)

for i in range(1,13): #12 images
    generate_binary_tiles(f'{base_path}/mask/{i}.png', 256, 256, 128, f'{base_path}/patch-binary')

print("Binary mask images are generated successfully.")


OSError: [Errno 45] Operation not supported: '/home/student'

## 2. Create dataset using tiles

Labels are in the folder names

In [ ]:
import os
import cv2
import numpy as np
import json

# Function to load annotations from a JSON file
def load_annotations(annotation_file):
    with open(annotation_file, 'r') as f:
        return json.load(f)

In [ ]:
annotation_files = [f'{base_path}/Tiles/instances_1_2_3.json',
                    f'{base_path}/Tiles/instances_4_5_6.json',
                    f'{base_path}/Tiles/instances_7_8_9.json',
                    f'{base_path}/Tiles/instances_10_11_12.json']
# Ensure the output directory exists
os.makedirs(f'{base_path}/Annotations', exist_ok=True)

In [ ]:
#label encouding (amnual)
building_code = [0,1]

colour_code = {'White':1,
                'Red':2,
                'Black':3,
                'Green':4,
                'Blue':5,
                'Brown':6,
                'Grey':7,
                'Orange':8,
                'Yellow':9}

texture_code = {'Rough':1,
                 'Average':2,
                 'Smooth':3}

score_code = {'100':10,
               '90':9,
               '80':8,
               '70':7,
               '60':6,
               '50':5
               }

material_code = {'Concrete':1,
                  'Concrete Slate':2,
                  'Concrete Tiles':3,
                  'Concrete Ballast':4,
                  'Metal':5,
                  'Metal Tile':6,
                  'Steel':7,
                  'Green':8,
                  'Tiles':9,
                  'Glass':10,
                  'Asphalt Shingles':11,
                  'Bitumen':12,
                  'EPDM':13,
                  'Slate':14,
                  'Stone Tile':15,
                  'Complex':16,
                  'Solar':17
               }
               

In [ ]:
images = []
building_masks =[]
colour_masks=[]
texture_masks=[]
score_masks=[]
materials_masks=[]
image_dir = 'data/Tiles'

for i,file in enumerate (annotation_files):
    annotation_data=load_annotations(file)
    for image_info in annotation_data['images']:
        image_path = os.path.join(image_dir, image_info['file_name'])
        # Load the original image
        image = cv2.imread(image_path)
        image = cv2.cvtColor(image,cv2.COLOR_BGR2RGB)
        if image is None:
            print(f"Failed to load original image at {image_path}")
        else:
            print(f"Successfully load original image at {image_path}")
            images.append(image)
            building_masks.append(np.zeros(image.shape[:2],int))#we only need one channel for masks
            colour_masks.append(np.zeros(image.shape[:2],int))#we only need one channel for masks
            texture_masks.append(np.zeros(image.shape[:2],int))#we only need one channel for masks
            score_masks.append(np.zeros(image.shape[:2],int))#we only need one channel for masks
            materials_masks.append(np.zeros(image.shape[:2],int))#we only need one channel for masks


    for annotation in annotation_data['annotations']:
        #get annotation values from the datafile
        id = annotation['id']
        image_idx = annotation['image_id']
        region = annotation['segmentation']
        area = annotation['area']
        bbox = annotation['bbox']
        a_colour = annotation['attributes']['Colours']
        a_texture = annotation['attributes']['Textures']
        a_score = annotation['attributes']['Score']
        a_materials = annotation['attributes']['Materials']
        a_occluded = annotation['attributes']['occluded']

        #Building masks
        polygon = np.reshape(region,(int(np.size(region)/2),2)).astype(np.int32).reshape((-1, 1, 2))#format change to opencv polygon x and y pairs
        building_masks[i*3+image_idx-1]  = cv2.fillPoly(building_masks[i*3+image_idx-1] ,[polygon],color=building_code[1])

        #Colour masks
        colour_masks[i*3+image_idx-1] = cv2.fillPoly(colour_masks[i*3+image_idx-1],[polygon],color=colour_code[a_colour])

        # #Texture masks
        texture_masks[i*3+image_idx-1] = cv2.fillPoly(texture_masks[i*3+image_idx-1],[polygon],color=texture_code[a_texture])

        # Score masks
        score_masks[i*3+image_idx-1] = cv2.fillPoly(score_masks[i*3+image_idx-1],[polygon],color=score_code[a_score])

        # Material masks
        materials_masks[i*3+image_idx-1] = cv2.fillPoly(materials_masks[i*3+image_idx-1],[polygon],color=material_code[a_materials])


In [ ]:
from PIL import Image
im = Image.fromarray
def generate_tiles_from_masks(image_idx, image_mask, tile_width, tile_height, step, output_folder):
    # Open the image
    image_mask = Image.fromarray(image_mask[:,:,0]*255)
    image_width, image_height= image_mask.size

    # Generate tiles
    for x in range(0, image_width - tile_width + 1, step):
        for y in range(0, image_height - tile_height + 1, step):
            # Define the bounding box for the current tile
            left = x
            upper = y
            right = left + tile_width
            lower = upper + tile_height
            bbox = (left, upper, right, lower)

            # Crop the image to the bounding box to create the tile
            tile = image_mask.crop(bbox)

            # Save the tile to the output folder
            tile.save(f"{output_folder}/{image_idx}_{x}_{y}.png") #image_id_x_y.png

    print(f"Tiles from image id {image_idx} are generated successfully.")

In [ ]:
for idx, mask in enumerate(building_masks): #12 images
    generate_tiles_from_masks(idx+1, mask, 100, 100, 50, "Annotations")


##  using the tiles version of the images and masks, we can now train a segmentation model